# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR\u00b2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. 

### Dataset Source
The dataset is described by a Croissant schema available at:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata and make available as Python object
dataset = mlc.Dataset(croissant_url)

# Print out dataset main information
meta = dataset.metadata
print(f"Name: {meta.name}")
print(f"Description: {meta.description}\n")
print(f"Identifier: {meta.identifier}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"License: {getattr(meta, 'license', 'N/A')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s for extraction and analysis.

In [ ]:
# List record sets and their fields by @id
record_sets = dataset.list_record_sets()
print("Available record sets and fields:")
for record_set in record_sets:
    print(f"- RecordSet @id: {record_set['@id']}")
    field_ids = [field['@id'] for field in record_set['fields']]
    print(f"  Fields (@id): {field_ids}\n")

## 3. Data Extraction
Load data from a selected record set into a DataFrame for analysis. 

**Note:** All accesses are by the record set and field `@id`s, following FAIR\u00b2's Croissant schema.

In [ ]:
# Choose an example record set @id from the overview above.
# We use the first non-empty record set found.
example_rs = None
for record_set in record_sets:
    if 'fields' in record_set and len(record_set['fields']) > 0:
        example_rs = record_set['@id']
        break
if not example_rs:
    raise Exception("No valid record set found in metadata!")
print(f"Using record set: {example_rs}\n")

# Extract records for this record set by @id
records = list(dataset.records(record_set=example_rs))
df = pd.DataFrame(records)
print(f"Loaded columns for record set {example_rs}:")
print(df.columns.tolist())

# Preview first 5 records
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing: filtering, normalization, and grouping using only column `@id`s. 

Replace the example field IDs with those found in the previous overview and extraction.

In [ ]:
# Pick a numeric field and a categorical field from the DataFrame columns
# For demonstration, try to find a likely numeric and group field automatically
import numpy as np

numeric_field = None
group_field = None
for col in df.columns:
    # Try to find a float or int column
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field = col
        break
# Try to find a groupable field (not numeric, not index)
for col in df.columns:
    if col != numeric_field and df[col].nunique() < len(df) // 2:
        group_field = col
        break

if numeric_field is not None:
    print(f"Using numeric field @id: {numeric_field}")
else:
    print("No numeric field found for EDA example!")
if group_field is not None:
    print(f"Using group field @id: {group_field}\n")

# Automatically pick threshold: mean if reasonable, else 0
if numeric_field is not None:
    threshold = df[numeric_field].dropna().mean() if len(df[numeric_field].dropna()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
    print(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group if possible
    if group_field is not None:
        grouped_df = filtered_df.groupby(group_field, dropna=True)[numeric_field].mean()
        print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or field relationships. This will plot the numeric field distribution and group averages if such fields are present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field histogram
if numeric_field is not None and len(df[numeric_field].dropna()) > 0:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.tight_layout()
    plt.show()

# Grouped barplot if group and numeric field exist
if numeric_field is not None and group_field is not None:
    grp = df[[group_field, numeric_field]].dropna()
    # Aggregate mean per group
    grp_avg = grp.groupby(group_field)[numeric_field].mean().reset_index()
    plt.figure(figsize=(8,4))
    sns.barplot(data=grp_avg, x=group_field, y=numeric_field)
    plt.title(f"Mean {numeric_field} grouped by {group_field}")
    plt.xticks(rotation=25)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
This notebook loaded and explored the FAIR\u00b2 mass spectrometry dataset using `mlcroissant`, showing dataset structure via Croissant '@id's, extracting the main record set, displaying fields, and performing basic EDA and visualization by referencing all data elements through their unique Croissant identifiers.